# 🧪 Laboratorio Práctico de **Pandas** (Aplicación de la Guía)
_Generado: 2025-11-02 05:46_

Este laboratorio te guía paso a paso para **aplicar** los conceptos de la guía de Pandas: lectura de datos, exploración, selección/filtrado, limpieza, transformaciones, agrupamientos, combinaciones, visualización y exportación.

**Datasets incluidos (ya guardados en el entorno):**
- `lab_ventas.csv` — ventas diarias por región/producto/vendedor, con algunos nulos y duplicados.
- `lab_clientes.csv` — catálogo de clientes con región y segmento.

> Recomendación: ejecuta las celdas en orden. En cada ejercicio hay una celda **"Tu solución aquí"** y luego una **"Posible solución"**.


## 🎯 Objetivos
- Practicar el flujo de análisis de datos con **Pandas** end‑to‑end.
- Ejercitar **limpieza**, **transformaciones**, **groupby**, **merge** y **visualización**.
- Exportar resultados a formatos comunes.


## 🚀 Preparación
Si usas un entorno fuera de este, asegúrate de tener instaladas las dependencias:

```bash
pip install pandas numpy matplotlib openpyxl
```


In [1]:
# Importaciones
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', 6)
pd.set_option('display.max_columns', None)


In [ ]:
# Cargar datasets del laboratorio
ventas_path = r'C:\Users\esteb\Downloads\n\lab_ventas.csv'
clientes_path = r'C:\Users\esteb\Downloads\n\lab_clientes.csv'

df_ventas = pd.read_csv(ventas_path, parse_dates=['fecha'])
df_clientes = pd.read_csv(clientes_path)

df_ventas.head(), df_clientes.head()


### Ejercicio 1 — Exploración inicial
1. Muestra `info()` y `describe()` (solo numéricas) de `df_ventas`.
2. ¿Cuántos nulos hay en `cantidad`?
3. ¿Hay filas duplicadas? ¿Cuántas?


In [ ]:
# Tu solución aquí


In [ ]:
# Posible solución
df_ventas.info()
display(df_ventas.describe(numeric_only=True))
display(df_ventas['cantidad'].isna().sum())
display(df_ventas.duplicated().sum())


### Ejercicio 2 — Limpieza de datos
1. Imputa los nulos de `cantidad` con la **mediana**.
2. Elimina las **filas duplicadas** completas.
3. Verifica nuevamente nulos y tamaño final del DataFrame.


In [ ]:
# Tu solución aquí


In [ ]:
# Posible solución
mediana = df_ventas['cantidad'].median()
df_ventas['cantidad'] = df_ventas['cantidad'].fillna(mediana)
antes = len(df_ventas)
df_ventas = df_ventas.drop_duplicates()
despues = len(df_ventas)
display({'nulos_cantidad': int(df_ventas['cantidad'].isna().sum()),
         'antes': antes, 'despues': despues})


### Ejercicio 3 — Transformaciones
1. Crea `ingreso` (si no existe) como `cantidad * precio_unitario`.
2. Crea `trimestre` a partir de `fecha`.
3. Crea `ticket_promedio` por fila como `ingreso / cantidad`.


In [ ]:
# Tu solución aquí


In [ ]:
# Posible solución
if 'ingreso' not in df_ventas.columns:
    df_ventas['ingreso'] = df_ventas['cantidad'] * df_ventas['precio_unitario']

df_ventas['trimestre'] = df_ventas['fecha'].dt.to_period('Q').astype(str)
df_ventas['ticket_promedio'] = df_ventas['ingreso'] / df_ventas['cantidad']
df_ventas.head()


### Ejercicio 4 — Selección y filtrado
1. Selecciona ventas de la **región 'Norte'** con `ingreso > 300`.
2. Muestra solo las columnas `fecha`, `region`, `vendedor`, `ingreso` de esos registros.


In [ ]:
# Tu solución aquí


In [ ]:
# Posible solución
filtro = (df_ventas['region'] == 'Norte') & (df_ventas['ingreso'] > 300)
df_filtrado = df_ventas.loc[filtro, ['fecha', 'region', 'vendedor', 'ingreso']]
df_filtrado.head()


### Ejercicio 5 — Agrupamientos y agregaciones
1. Calcula el **ingreso total** por `region` ordenado descendente.
2. Calcula, por `producto`, `cantidad_total`, `ingreso_promedio` y `transacciones`.
3. ¿Qué `vendedor` tiene el **mayor ingreso total**?


In [ ]:
# Tu solución aquí


In [ ]:
# Posible solución
ingreso_por_region = (df_ventas.groupby('region', as_index=False)['ingreso']
                      .sum().sort_values('ingreso', ascending=False))
display(ingreso_por_region)

agg_producto = df_ventas.groupby('producto').agg(
    cantidad_total=('cantidad', 'sum'),
    ingreso_promedio=('ingreso', 'mean'),
    transacciones=('producto', 'count')
).reset_index()
display(agg_producto)

top_vendedor = (df_ventas.groupby('vendedor', as_index=False)['ingreso'].sum()
                .sort_values('ingreso', ascending=False).head(1))
display(top_vendedor)


### Ejercicio 6 — Unión y combinación de DataFrames
1. Une (`merge`) `df_ventas` con `df_clientes` por `id_cliente`.
2. Calcula el **ingreso total por segmento** y ordénalo descendente.
3. ¿Qué **segmento** aporta más ingreso?


In [ ]:
# Tu solución aquí


In [ ]:
# Posible solución
df_comb = pd.merge(df_ventas, df_clientes, on='id_cliente', how='left')
ingreso_segmento = (df_comb.groupby('segmento', as_index=False)['ingreso']
                    .sum().sort_values('ingreso', ascending=False))
display(ingreso_segmento)


### Ejercicio 7 — Visualización con `pandas.plot()` y matplotlib
1. Histograma de la distribución de `ingreso`.
2. Barras del ingreso total por `region`.
> **Reglas**: usa **matplotlib**, **una gráfica por figura** y **no cambies colores/estilos** manualmente.


In [ ]:
# Tu solución aquí


In [ ]:
# Posible solución
# Histograma
plt.figure()
df_ventas['ingreso'].plot(kind='hist', bins=20, edgecolor='black')
plt.title('Distribución de ingresos')
plt.xlabel('Ingreso')
plt.ylabel('Frecuencia')
plt.show()

# Barras por región
ingreso_region_plot = (df_ventas.groupby('region', as_index=False)['ingreso']
                       .sum().sort_values('ingreso', ascending=False))
plt.figure()
ingreso_region_plot.set_index('region')['ingreso'].plot(kind='bar')
plt.title('Ingreso total por región')
plt.xlabel('Región')
plt.ylabel('Ingreso total')
plt.show()


### Ejercicio 8 — Exportación
1. Exporta `ingreso_por_region` a **CSV** y `ingreso_segmento` a **Excel** (si aplican a tus resultados).
2. Verifica que los archivos existan listando el directorio de trabajo.


In [ ]:
# Tu solución aquí


In [ ]:
# Posible solución
# Reutilizamos las variables si existen, de lo contrario las calculamos rápidamente
try:
    ingreso_por_region
except NameError:
    ingreso_por_region = (df_ventas.groupby('region', as_index=False)['ingreso']
                          .sum().sort_values('ingreso', ascending=False))

try:
    ingreso_segmento
except NameError:
    df_comb = pd.merge(df_ventas, df_clientes, on='id_cliente', how='left')
    ingreso_segmento = (df_comb.groupby('segmento', as_index=False)['ingreso']
                        .sum().sort_values('ingreso', descending=False))


In [1]:
# (Corrección) Exportación con sort correcto
df_comb = pd.merge(df_ventas, df_clientes, on='id_cliente', how='left')
ingreso_por_region = (df_ventas.groupby('region', as_index=False)['ingreso']
                      .sum().sort_values('ingreso', ascending=False))
ingreso_segmento = (df_comb.groupby('segmento', as_index=False)['ingreso']
                    .sum().sort_values('ingreso', ascending=False))

csv_out = "lab_ingreso_por_region.csv"
xlsx_out = "lab_ingreso_por_segmento.xlsx"

ingreso_por_region.to_csv(csv_out, index=False)

with pd.ExcelWriter(xlsx_out) as writer:
    ingreso_segmento.to_excel(writer, index=False, sheet_name='ingreso_segmento')

import os
sorted([p for p in os.listdir('.') if p.endswith(('.csv', '.xlsx'))])


NameError: name 'pd' is not defined

## ✅ Checklist de verificación
- [ ] Exploraste estructura y estadísticas de los datos.
- [ ] Imputaste nulos y eliminaste duplicados.
- [ ] Creaste nuevas columnas derivadas (fecha → trimestre, ticket, etc.).
- [ ] Aplicaste `groupby` con múltiples agregaciones.
- [ ] Combinaste tablas con `merge` y analizaste por segmento.
- [ ] Generaste al menos **2 gráficas** con `pandas.plot()`.
- [ ] Exportaste resultados a CSV/Excel.

## 🧰 Siguientes ideas
- Analiza ventas por **día de la semana** o **festivos**.
- Usa `resample('M')` para mensualizar ingresos.
- Crea categorías con `pd.cut` o `np.where` y compáralas por región.
